# 10 CV Baseline

Run this notebook after `00_data_loading_and_split.ipynb`. It evaluates a restart-safe constant-velocity trajectory baseline and saves metrics, plots, and predictions to disk.


In [1]:
import gc
import json
import sys
from pathlib import Path

import torch

gc.collect()
try:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

PROJECT_ROOT = Path.home() / "Documents/Thesis"
PIPELINE_ROOT = PROJECT_ROOT / "08_model_training_pipeline"
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from training.notebook_workflow import (
    CNNGNNLSTMTrajectoryPredictor,
    CNNGNNLSTMTransformerTrajectoryPredictor,
    CNNGNNTransformerTrajectoryPredictor,
    CNNLSTMTrajectoryPredictor,
    GoalOnlyTrajectoryDataset,
    LSTMGoalTrajectoryPredictor,
    ScanGoalTrajectoryDataset,
    ScanGraphTrajectoryDataset,
    collate_goal_only,
    collate_scan,
    collate_scan_graph,
    device_from_flag,
    evaluate_trajectory_model,
    load_or_build_shared_artifacts,
    make_dataloaders,
    prepare_result_dirs,
    run_constant_velocity_baseline,
    save_final_trajectory_evaluation,
    set_seed,
    timestamp_tag,
    train_trajectory_model,
)


In [2]:
SEED = 42
PAST_LEN = 10
FUTURE_LEN = 5
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAX_SAMPLES = None
USE_CPU = False

BATCH_SIZE = 64
EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4

GOAL_DIM = 13
NODE_DIM = 12
EDGE_DIM = 8
HIDDEN_DIM = 128
GRAPH_HIDDEN = 96
DROPOUT = 0.10
MSG_PASSES = 2
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_FF = 256

device = device_from_flag(USE_CPU)
print("Device:", device)


Device: cuda


In [3]:
set_seed(SEED)
streams, sample_table, split_info, sample_table_path, split_path = load_or_build_shared_artifacts(
    past_len=PAST_LEN,
    future_len=FUTURE_LEN,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
)
print("Sample table:", sample_table_path)
print("Split path:", split_path)
print("Split strategy:", split_info["split_strategy"])
print("Episode count:", split_info["episode_count"])
print("Train / Val / Test samples:", len(split_info["train_indices"]), len(split_info["val_indices"]), len(split_info["test_indices"]))
print("Train episodes:", split_info["train_episode_ids"])
print("Val episodes:", split_info["val_episode_ids"])
print("Test episodes:", split_info["test_episode_ids"])


Sample table: /home/basudeo/Documents/Thesis/08_model_training_pipeline/results/train_ready/sample_table_seed42_past10_future5.json
Split path: /home/basudeo/Documents/Thesis/08_model_training_pipeline/results/splits/trajectory_split_seed42_past10_future5.json
Split strategy: episode
Episode count: 6
Train / Val / Test samples: 29379 6195 8967
Train episodes: ['run_model_20260511_220103', 'run_model_20260511_210809', 'run_model_20260511_213251', 'run_model_20260511_221726']
Val episodes: ['run_model_20260511_202309']
Test episodes: ['run_model_20260511_222947']


In [4]:
MODEL_SLUG = "cv_baseline"
TIMESTAMP = timestamp_tag()
result_dir, weight_dir, plot_dir = prepare_result_dirs(MODEL_SLUG)
final_metrics = run_constant_velocity_baseline(
    streams=streams,
    sample_table=sample_table,
    split_info=split_info,
    model_slug=MODEL_SLUG,
    timestamp=TIMESTAMP,
    split_path=split_path,
    sample_table_path=sample_table_path,
    result_dir=result_dir,
    plot_dir=plot_dir,
)
print(json.dumps(final_metrics, indent=2))


{
  "ADE": 0.12070605158805847,
  "FDE": 0.17679579555988312,
  "RMSE": 1.1617037057876587,
  "loss": 0.12070605158805847,
  "model_slug": "cv_baseline",
  "timestamp": "20260512_180822",
  "split_path": "/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/splits/trajectory_split_seed42_past10_future5.json",
  "sample_table_path": "/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/train_ready/sample_table_seed42_past10_future5.json",
  "prediction_export_path": "/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/cv_baseline/20260512_180822_trajectory_predictions.npz",
  "overlay_paths": [
    "/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/cv_baseline/plots/20260512_180822_trajectory_overlay_00.png",
    "/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/cv_baseline/plots/20260512_180822_trajectory_overlay_01.png",
    "/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/cv_baseline/plots/2026